# 04.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/05-scaling/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

spend5 = np.array([200, 400, 600, 800, 1000], dtype=float)

abc = pd.DataFrame({"orders": [2, 12, 2], "spend": [500, 520, 900]}, index=["A", "B", "C"])

3,317 customers, churn rate 43.6%


train 2,487 / test 830


## Exercise 1

Mean = 20/5 = 4. Squared deviations: 9, 4, 1, 0, 36 → sum 50 → variance 50/5 = 10 → σ = 3.162.

z = (1−4)/3.162 = −0.949, −0.632, −0.316, 0, **1.897**. Min-max: (x−1)/9 = 0, 0.111, 0.222, 0.333, **1**.

The customer with 10 orders is the extreme under both. Under min-max the other four are squeezed into 0–0.33.

In [2]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
o = np.array([1, 2, 3, 4, 10.]).reshape(-1, 1)
print("z:      ", StandardScaler().fit_transform(o).ravel().round(3))
print("min-max:", MinMaxScaler().fit_transform(o).ravel().round(3))

z:       [-0.949 -0.632 -0.316  0.     1.897]
min-max: [0.    0.111 0.222 0.333 1.   ]


## Exercise 2

In [3]:
for name, f in [("raw spend", lambda s: s), ("log1p(spend)", np.log1p)]:
    mm = MinMaxScaler().fit(f(X_train[["spend"]]))
    scaled = mm.transform(f(X_train[["spend"]])).ravel()
    print(f"{name:<14} share of customers below 0.01: {(scaled < 0.01).mean():.1%}   median {np.median(scaled):.3f}")

raw spend      share of customers below 0.01: 74.0%   median 0.004
log1p(spend)   share of customers below 0.01: 0.0%   median 0.473


On raw spend, the wholesalers set the maximum, so three quarters of customers are squashed below 0.01 (median 0.004): KNN would see almost no difference between a £100 and a £1,000 customer. After `log1p`, customers spread across the range. Give KNN the log version.

## Exercise 3

In [4]:
from sklearn.neighbors import KNeighborsClassifier
right = StandardScaler().fit(X_train)
wrong_tr, wrong_te = StandardScaler().fit(X_train), StandardScaler().fit(X_test)
knn = KNeighborsClassifier(15)
print(f"right: {knn.fit(right.transform(X_train), y_train).score(right.transform(X_test), y_test):.1%}")
print(f"wrong: {knn.fit(wrong_tr.transform(X_train), y_train).score(wrong_te.transform(X_test), y_test):.1%}")
print("train vs test spend mean:", round(X_train['spend'].mean()), round(X_test['spend'].mean()))

right: 64.5%
wrong: 64.6%
train vs test spend mean: 1510 1773


With 830 test customers the two scalers learn similar means (spend £1,510 vs £1,773), so the two scores are within a tenth of a point here. But the test data has been put on a *slightly different ruler* from the training data, and the smaller or more unusual the batch, the bigger the error. For a single live customer it breaks completely: one value has a standard deviation of 0, so the scaled value is undefined (scikit-learn returns 0 for everything). Always reuse the training scaler.

## Exercise 4

In [5]:
def log_some(X):
    return X.assign(**{c: np.log1p(X[c]) for c in ["orders", "spend", "products"]})

sc = StandardScaler().fit(log_some(X_train))
A, B = sc.transform(log_some(X_train)), sc.transform(log_some(X_test))
for k in [5, 15, 51]:
    print(f"k={k:>2}: log + scaled {KNeighborsClassifier(k).fit(A, y_train).score(B, y_test):.1%}")

k= 5: log + scaled 61.7%
k=15: log + scaled 65.8%
k=51: log + scaled 67.2%


Compared with plain scaling in the lesson (61.4%, 64.5%, 67.8%), log-then-scale gains at k = 5 and 15 and is slightly lower at k = 51. None of the KNN variants beats the scaled logistic regression (67.5%). Log-then-scale is the right *default* for distance-based models on money-like columns, because it makes features comparable in size *and* evenly spread, but it is not magic; always compare on validation data. Notice the order of operations: transform (log), then scale, both fitted on training rows. Lesson 04.8 puts both steps in one pipeline.